In [ ]:
!git clone https://github.com/boracandan/Algoverse_Truth_Directions_Research.git


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

print("✓ All imports complete")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
save_dir = "/content/drive/MyDrive/algoverse_results"
DATA_DIR = "/content/Algoverse_Truth_Directions_Research/CoT_datasets/lexically_cleaned"
os.makedirs(save_dir, exist_ok=True)
csv_path = f"{save_dir}/results_database.csv"

print(f"Save directory: {save_dir}")
print(f"Data directory: {DATA_DIR}")

# Verify dataset exists
if os.path.exists(DATA_DIR):
    print(f"✓ Dataset found")
    print(f"  Files: {os.listdir(DATA_DIR)[:5]}")
else:
    print(f"✗ Dataset NOT found at {DATA_DIR}")

In [ ]:
print("Cleaning GPU...")
gc.collect()
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print("✓ Tokenizer loaded")

print("Loading model ")
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()

print(f"✓ Model loaded")
print(f"  Device: {model.device}")
print(f"  GPU memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"  Layers: {model.config.num_hidden_layers}")

In [ ]:
print("Loading COT datasets...")
DATA_DIR = "/mnt/user-data/uploads"

F0_train = pd.read_csv(f"{DATA_DIR}/F0_train.csv")
F0_test = pd.read_csv(f"{DATA_DIR}/F0_test.csv")
F1_train = pd.read_csv(f"{DATA_DIR}/F1_train.csv")
F1_test = pd.read_csv(f"{DATA_DIR}/F1_test.csv")
F2_train = pd.read_csv(f"{DATA_DIR}/F2_train.csv")
F2_test = pd.read_csv(f"{DATA_DIR}/F2_test.csv")
F3_train = pd.read_csv(f"{DATA_DIR}/F3_train.csv")
F3_test = pd.read_csv(f"{DATA_DIR}/F3_test.csv")
F4_train = pd.read_csv(f"{DATA_DIR}/F4_train.csv")
F4_test = pd.read_csv(f"{DATA_DIR}/F4_test.csv")
F5_train = pd.read_csv(f"{DATA_DIR}/F5_train.csv")
F5_test = pd.read_csv(f"{DATA_DIR}/F5_test.csv")
A1_train = pd.read_csv(f"{DATA_DIR}/A1_train.csv")
A1_test = pd.read_csv(f"{DATA_DIR}/A1_test.csv")
A2_train = pd.read_csv(f"{DATA_DIR}/A2_train.csv")
A2_test = pd.read_csv(f"{DATA_DIR}/A2_test.csv")
A3_train = pd.read_csv(f"{DATA_DIR}/A3_train.csv")
A3_test = pd.read_csv(f"{DATA_DIR}/A3_test.csv")

tasks_dict = {
    "F0": (F0_train, F0_test), "F1": (F1_train, F1_test), "F2": (F2_train, F2_test),
    "F3": (F3_train, F3_test), "F4": (F4_train, F4_test), "F5": (F5_train, F5_test),
    "A1": (A1_train, A1_test), "A2": (A2_train, A2_test), "A3": (A3_train, A3_test),
}
task_names = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]

print("✓ All COT datasets loaded:")
for name, (tr, te) in tasks_dict.items():
    print(f"    {name}: {len(tr)} train / {len(te)} test")

In [ ]:
def activations_all_layers(model, statements, batch_size=4):
    statements = list(statements)
    num_layers = model.config.num_hidden_layers + 1
    activations_by_layer = [[] for _ in range(num_layers)]

    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer_idx, layer_hidden in enumerate(outputs.hidden_states):
            final_token = layer_hidden[:, -1, :]
            activations_by_layer[layer_idx].append(final_token.cpu())

        del outputs, inputs
        torch.cuda.empty_cache()

    return [torch.cat(layer_acts, dim=0) for layer_acts in activations_by_layer]


def train_probe(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels
    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(1000):
        optimizer.zero_grad()
        loss = loss_fn(probe(X_train_t).squeeze(-1), y_train_t)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()

    auroc = roc_auc_score(y_test, test_logits)
    w = probe.weight.detach().cpu().numpy().flatten()
    return w, train_mean, auroc


def train_all_layers(train_acts, test_acts, y_train, y_test):
    num_layers = len(train_acts)
    layer_results = {}
    for layer_idx in range(num_layers):
        X_train = train_acts[layer_idx]
        X_test = test_acts[layer_idx]
        X_train_np = torch.stack(list(X_train)).float().numpy()
        variance = X_train_np.var(axis=0) + 1e-6
        w, train_mean, auroc = train_probe((X_train, X_test), (y_train, y_test))
        layer_results[layer_idx] = {
            "auroc": auroc,
            "weights": w,
            "variance": variance,
            "train_mean": train_mean,
        }
        if layer_idx % 8 == 0:
            print(f"    Layer {layer_idx}: AUROC = {auroc:.4f}")
    return layer_results


def write_indomain_rows_cot(task_name, layer_results, csv_path, train_condition="cot-zero-shot"):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        mask = ~((df["train_task"] == task_name) & 
                 (df["test_task"] == task_name) & 
                 (df["model"] == "deepseek-r1-distill-8b") &
                 (df["train_condition"] == train_condition))
        df = df[mask]
    else:
        df = pd.DataFrame(columns=["train_task","test_task","train_condition","test_condition","layer","model","auroc"])

    new_rows = []
    for layer_idx, data in layer_results.items():
        new_rows.append({
            "train_task": task_name,
            "test_task": task_name,
            "train_condition": train_condition,
            "test_condition": train_condition,
            "layer": layer_idx,
            "model": "deepseek-r1-distill-8b",
            "auroc": data["auroc"],
        })

    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"  ✓ Saved {len(new_rows)} layers for {task_name} ({train_condition})")

print("✓ All functions defined")

In [ ]:
task_name = "A1"
print(f"\nProcessing {task_name} (COT)...")
train_df, test_df = tasks_dict[task_name]

# Use extracted_statement_texts column for COT
train_acts = activations_all_layers(model, train_df["statement"], batch_size=4)
test_acts = activations_all_layers(model, test_df["statement"], batch_size=4)
results_A1_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"], probe_device=device)

del train_acts, test_acts
if torch.cuda.is_available():
    torch.cuda.empty_cache()

write_indomain_rows_cot(task_name, results_A1_cot, csv_path, train_condition="cot")
print(f"✓ Done processing {task_name}")